In [3]:
import sys
import os
import numpy as np
from scipy import stats
import time
import pandas as pd
import itertools

sys.path.append(os.path.abspath('..'))

from LSM.stochastic_processes import GeometricBrownianMotion
from LSM.payoffs import AmericanOption
from LSM.regression_bases import LaguerrePolynomials
from LSM.algorithms import LeastSquaresMonteCarlo

%load_ext autoreload
%autoreload 2

In [6]:
#(1) BSM Benchmark: Black-Scholes-Merton Model for European Options (Closed Form - Table 1 & Sanity Check)
def BSM(S0: float, K: float, T: float, r: float, q: float, sigma: float, option_type: str) -> float:
    """
    Returns the BSM price estimation of a European Option
    """
    option_type = option_type.lower()
    if option_type not in ['call', 'put']:
        raise ValueError("option_type must be either 'call' or 'put'.")

    d1 = (np.log(S0/K) + (r - q + 0.5*(sigma**2))*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type == 'put':
        price = np.exp(-r*T) * K * stats.norm.cdf(-d2) - np.exp(-q*T) * S0 * stats.norm.cdf(-d1)
    else:
        price = np.exp(-q*T) * S0 * stats.norm.cdf(d1) - np.exp(-r*T) * K * stats.norm.cdf(d2)

    stdev = 0.0 #BSM is an analytical formula pricing European Options

    return price, stdev

In [17]:
# Sanity check: an American call without dividend (q=0) is equivalent to a 
# European call. Compare our LSM price with the BSM European call price.
# Accuracy, runtime, (memory storage)

# Sample test parameters
S0 = 36
K = 40
r = 0.06
q = 0.0
sigma = 0.2
T = 1.0
n_steps = 50
n_paths = 10000

# BSM Sanity Check: (American Call Option = European Call)
start_bsm = time.time()
bsm_call, _ = BSM(S0, K, T, r, q, sigma, option_type = "call")
end_bsm = time.time()
print(f"BSM European Call Price: {bsm_call}.")

call_payoff = AmericanOption(strike=K, option_type = "call")
gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
laguerre_basis = LaguerrePolynomials(degree=3)

lsm_eng = LeastSquaresMonteCarlo(
    process = gbm,
    payoff_function = call_payoff,
    basis_function=laguerre_basis
    )
start_lsm = time.time()
lsm_call, lsm_se = lsm_eng.pricer(T = T, n_steps = n_steps, n_paths = n_paths)
end_lsm = time.time()
print(f"LSM American Call Price {lsm_call}.")
print(f"Absolute Error between BSM and LSM on Call: {abs(bsm_call - lsm_call)}")

print(f"BSM Runtime: {end_bsm - start_bsm}")
print(f"LSM Runtime: {end_lsm - start_lsm}")

BSM European Call Price: (2.1737264482268923, 0.0).
LSM American Call Price 2.19333868790882.
Absolute Error between BSM and LSM on Call: [0.01961224 2.19333869]
BSM Runtime: 0.024382829666137695
LSM Runtime: 0.05225706100463867


In [9]:
# TODO: Use the American put numerical example in Longstaff, Schartz (2001) pp. 126-128.
# Goals: 
# a) replicate their results; 
# b) compare with other benchmarks and libraries (e.g. CRR Binomial Tree),
# focusing on accuracy (prices), speed (%timeit) and memory usage
# [CREATE TESTS USING OTHER LIBRARIES FIRST!]


S0 = 36.0
r = 0.06
q = 0.0
sigma = 0.2
K = 40.0
T = 1.0
n_steps = 50
n_paths = 10000


gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
put_payoff = AmericanOption(strike=K, option_type="call") # "put"
laguerre_basis = LaguerrePolynomials(degree=3)


lsm_engine = LeastSquaresMonteCarlo(
    process=gbm, 
    payoff_function=put_payoff, 
    basis_function=laguerre_basis
)


time_grid, paths = gbm.simulate(T=T, n_steps=n_steps, n_paths=n_paths)
print(f"1. GBM paths shape: {paths.shape}  --> Expect ({n_paths}, {n_steps + 1})")

payoff_at_maturity = put_payoff(paths[:, -1])
print(f"2. Payoff shape: {payoff_at_maturity.shape}  --> Expect: ({n_paths},)")

test_X = paths[:100, -1]
test_Y = payoff_at_maturity[:100]
beta = laguerre_basis.fit(X=test_X, Y=test_Y)
print(f"3. beta fitted on Laguerre basis \n{beta}  --> Expect: ")


price = lsm_engine.pricer(T=T, n_steps=n_steps, n_paths=n_paths)
print(f"4. LSM price: {price}  --> Expect: ")

1. GBM paths shape: (10000, 51)  --> Expect (10000, 51)
2. Payoff shape: (10000,)  --> Expect: (10000,)
3. beta fitted on Laguerre basis 
[3.14904237e+01 2.33298257e+00 1.02057708e-01 1.46998416e-03]  --> Expect: 
4. LSM price: (2.228240968012633, 0.04233258125989479)  --> Expect: 


In [18]:
#(2) Longstaff-Schwartz-Method (our implementation)
def LSM(S0, K, T, r, q, sigma, n_steps, n_paths, option_type='put'):
    gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
    put_payoff = AmericanOption(strike=K, option_type="put")
    laguerre_basis = LaguerrePolynomials(degree=3)


    lsm_engine = LeastSquaresMonteCarlo(
        process=gbm, 
        payoff_function=put_payoff, 
        basis_function=laguerre_basis
    )

    price = lsm_engine.pricer(T=T, n_steps=n_steps, n_paths=n_paths)[0]
    stdev = lsm_engine.pricer(T=T, n_steps=n_steps, n_paths=n_paths)[1]

    return price, stdev

In [10]:
#(3) Binomial Tree Benchmark:
def BTM(S0, K, T, r, q, sigma, n_steps, option_type = 'put'):
    return 2

In [25]:
#(4) Finite Difference Benchmark: 
# Import the existed Package of Finite Difference Method from QuantLib
import QuantLib as ql

def FDM(S0, K, T, r, q, sigma, n_steps, n_paths, option_type = "put"):
    """
    Utilize Finite-Difference Method to estimate the price of American Put Option
    FDM is directly imported from QuantLib
    """
    # Time Setting (today, expiry)
    TD = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = TD
    expiry = TD + ql.Period(int(T * 365), ql.Days)

    # Market Information
    spot = ql.QuoteHandle(ql.SimpleQuote(S0))
    inv = ql.YieldTermStructureHandle(ql.FlatForward(TD, r, ql.Actual365Fixed())) # risk-free rate
    div = ql.YieldTermStructureHandle(ql.FlatForward(TD, q, ql.Actual365Fixed())) # dividend rate
    vol = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(TD, ql.NullCalendar(), sigma, ql.Actual365Fixed()))

    # Black-Scholes-Merton Model
    bsm = ql.BlackScholesMertonProcess(spot, div, inv, vol)

    # Option Set-up
    if option_type.lower() == 'put':
        cp = ql.Option.Put
    else:
        cp = ql.Option.Call
    payoff = ql.PlainVanillaPayoff(cp, K)
    exercise = ql.AmericanExercise(TD, expiry)
    option = ql.VanillaOption(payoff, exercise)

    # Crank-Nicolson Finite-Difference Method
    fdm = ql.FdBlackScholesVanillaEngine(
        bsm,
        n_steps,
        n_paths
    )
    option.setPricingEngine(fdm)
    price = option.NPV()
    stdev = 0.0 #Note FDM has an analytical formula of pricing.


    return price, stdev


In [14]:
#(4) Comparison Table for LSM Accuracy and RunTime （需要验证）
# Assume here it is always American Put Option
# The Benchmark must be the estimated price by a highly stable and accurate model,
    #i.e., Binominal Tree Method

def comparison(model, name: str, benchmark_price, **kwargs):
    """
    Take all existed models (BSM, LSM, ...) to compute prices
    Compare Accuracy and RunTime of our LSM with other models
    Return a Comparison Table
    """

    start = time.time()
    price = model(**kwargs)
    end = time.time()

    runtime = end - start
    abs_error = abs(price - benchmark_price)
    #rel_error = abs_error / benchmark_price

    Ctable = {
        "Model": name,
        "Price": price,
        "Absolute Error": abs_error,
        "RunTime": runtime
    }

    return Ctable
 


In [ ]:
#Parameters 
cargs = dict(S0=S0, K=K, T=T, r=r, q=q, sigma=sigma, option_type = 'put')

#Benchmark
benchmark_price = LSM(**cargs, n_steps=n_steps, n_paths=n_paths, degree=3)

#Realization of Comparison Table
models = []

models.append(
    comparison(LSM, "Least Square Monte Carlo", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths, degree=3))
models.append(
    comparison(BTM, "Binomial Tree Model", benchmark_price, **cargs, n_steps=n_steps))
models.append(
    comparison(FDM, "Finite Difference Method", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths))
models.append(
    comparison(BSM, "Black Scholes Model", benchmark_price, **cargs)
)

#Convert into a Data Frame
df = pd.DataFrame(models)

print(df)

In [31]:
# (Part 1) Sanity Check: Re-Show Table 1 from Longstaff-Schwartz Paper
# Parameters: (fixed) K, r, q, n_steps, n_paths; S0, sigma, T (change in order)
fixed_param = {
    "K": 40,
    "r": 0.06,
    "q": 0.0,
    "n_steps": 50,
    "n_paths": 10000
}

S0_list = [36, 38, 40, 42, 44]
sigma_list = [0.20, 0.40]
Time_list = [1, 2]


# Results from Table 1 (American Put Option Price by FDM)
FDM_Table1 = {
    (36, 0.20, 1): 4.478, (36, 0.20, 2): 4.840,
    (36, 0.40, 1): 7.101, (36, 0.40, 2): 8.508,
    (38, 0.20, 1): 3.250, (38, 0.20, 2): 3.745,
    (38, 0.40, 1): 6.148, (38, 0.40, 2): 7.670,
    (40, 0.20, 1): 2.314, (40, 0.20, 2): 2.885,
    (40, 0.40, 1): 5.312, (40, 0.40, 2): 6.920,
    (42, 0.20, 1): 1.617, (42, 0.20, 2): 2.212,
    (42, 0.40, 1): 4.582, (42, 0.40, 2): 6.248,
    (44, 0.20, 1): 1.110, (44, 0.20, 2): 1.690,
    (44, 0.40, 1): 3.948, (44, 0.40, 2): 5.647,
}

# Results from Table 1 (American Put Option Price by LSM)
LSM_Table1 = {
    (36, 0.20, 1): (4.472, 0.010), (36, 0.20, 2): (4.821, 0.012),
    (36, 0.40, 1): (7.091, 0.020), (36, 0.40, 2): (8.488, 0.024),
    (38, 0.20, 1): (3.244, 0.009), (38, 0.20, 2): (3.735, 0.011),
    (38, 0.40, 1): (6.139, 0.019), (38, 0.40, 2): (7.669, 0.022),
    (40, 0.20, 1): (2.313, 0.019), (40, 0.20, 2): (2.879, 0.010),
    (40, 0.40, 1): (5.308, 0.018), (40, 0.40, 2): (6.921, 0.022),
    (42, 0.20, 1): (1.617, 0.007), (42, 0.20, 2): (2.206, 0.010),
    (42, 0.40, 1): (4.588, 0.017), (42, 0.40, 2): (6.243, 0.021),
    (44, 0.20, 1): (1.118, 0.007), (44, 0.20, 2): (1.675, 0.009),
    (44, 0.40, 1): (3.957, 0.017), (44, 0.40, 2): (5.622, 0.021), 
}

# Table 1 Replication using our own LSM engine
def table1_replication(LSM_engine, FDM_engine, iterations = 25, seed = 66):
    """
    Here we test our own LSM engine to replicate Table 1 from LongStaff-Schwartz Paper by iterations times,
        take "average price, standard deviation" and compare with Table 1 from paper
    Input:
        LSM_engine: our own implementation of LSM algorithm
            LSM(S0, K, T, r, q, sigma, n_steps, n_paths, degree, option_type='put')
        FDM_engine: the imported FDM algorithm from QuantLib
             FDM(S0, K, T, r, q, sigma, n_steps, n_paths, option_type = "put")
        iterations: as it states, the iteration times to reduce randomness
        seed: randomness
    Return:
        dataframe: a DataFrame with the same format of Paper Table 1
    """
    # fixed parameters
    K = fixed_param["K"]
    r = fixed_param["r"]
    q = fixed_param["q"]
    n_steps = fixed_param["n_steps"]
    n_paths = fixed_param["n_paths"]
    # storage list
    results = [] # Replication Table 1
    comparison = [] # Comparison Table

    # simulation
    for S0 in S0_list:
        for sigma in sigma_list:
            for T in Time_list:
                N_steps = n_steps * T
                feature = (S0, sigma, T)

                # LSM engine iterations
                lsm_prices = []
                for i in range(iterations):
                    np.random.seed(seed + i)
                    price, _ = LSM_engine(
                        S0 = S0, K = K, T = T, r = r, q = q, 
                        sigma = sigma, n_steps = N_steps, n_paths = n_paths, )
                    lsm_prices.append(price)

                lsm_price = np.mean(lsm_prices)
                lsm_std = np.std(lsm_prices, ddof = 1)

                # FDM engine iterations 
                fdm_price, _ = FDM_engine(
                    S0 = S0, K = K, T = T, r = r, q = q,
                    sigma = sigma, n_steps = 100, n_paths = 400)

                # BSM European Put Option Price
                BSM_price, _ = BSM(S0, K, T, r, q, sigma, option_type = "put")

                # Early Exercise Value: AmericanPut - EuropeanPut
                fdm_early_val = fdm_price - BSM_price
                lsm_early_val = lsm_price - BSM_price

                paper_lsm_val, paper_lsm_se = LSM_Table1[feature]

                # Accuracy Comparison
                diff_price_val = lsm_price - fdm_price
                diff_early_val =  lsm_early_val - fdm_early_val
                within_2se = abs(diff_early_val) <= 2 * lsm_std

                results.append({
                    "S0": S0,
                    "sigma": sigma,
                    "T": T,
                    #FDM vals
                    "Finite Difference American": fdm_price,
                    "Closed form European": BSM_price,
                    "Early exercise Value (FDM)": fdm_early_val,
                    #LSM simulated vals
                    "Simulated American": lsm_price,
                    "(s.e.)": lsm_std,
                    "Closed form European": BSM_price,
                    "Early exercise Value (LSM)": lsm_early_val,
                    # Difference between two methods
                    "Difference in early exercise value": diff_early_val
                })

                comparison.append({
                    "S0": S0,
                    "sigma": sigma,
                    "T": T,
                    # FDM benchmark
                    "FDM Price": round(fdm_price, 3),
                    "FDM early exercise": round(fdm_early_val, 3),
                    # LSM engine
                    "LSM Price": round(lsm_price, 3),
                    "LSM s.e.": round(lsm_std, 3),
                    "LSM early exercise": round(lsm_early_val, 3),
                    # Paper LSM
                    "Paper LSM": paper_lsm_val,
                    "Paper s.e.": paper_lsm_se,
                    # Accuracy
                    "Difference in Vals": diff_price_val,
                    "Difference in Early Exercise": diff_early_val,
                    "Whether within 2 s.e.": "yes" if within_2se else "no"
                })
    return pd.DataFrame(comparison)
                    
# generate Comparison Table between our replication and original Table 1
if __name__ == "__main__":
    dataframe = table1_replication(LSM, FDM, iterations = 25)

    print("\n---------------- Table 1 Replication Comparison ----------------")
    print(dataframe.to_string(index = False))

    print("\n---------------- Summary ----------------")
    print(f"Mean |Difference in Vals|: {dataframe['Difference in Vals'].abs().mean():.4f}")
    print(f"Max |Difference in Vals|: {dataframe['Difference in Vals'].abs().max():.4f}")
    within = (dataframe["Whether within 2 s.e."] == 'yes').sum()
    print(f"Whether within 2 s.e.: {within}/{len(dataframe)}")
    print(f"|accuracy| < 0.03: {(dataframe['Difference in Vals'].abs() < 0.03).sum()}/{len(dataframe)}")

    dataframe.to_csv("Table1_replication_comparison.csv", index = False)


---------------- Table 1 Replication Comparison ----------------
 S0  sigma  T  FDM Price  FDM early exercise  LSM Price  LSM s.e.  LSM early exercise  Paper LSM  Paper s.e.  Difference in Vals  Difference in Early Exercise Whether within 2 s.e.
 36    0.2  1      4.482               0.638      4.477     0.024               0.633      4.472       0.010           -0.005074                     -0.005074                   yes
 36    0.2  2      4.840               1.077      4.839     0.034               1.076      4.821       0.012           -0.001599                     -0.001599                   yes
 36    0.4  1      7.105               0.394      7.088     0.062               0.377      7.091       0.020           -0.017367                     -0.017367                   yes
 36    0.4  2      8.507               0.807      8.559     0.078               0.859      8.488       0.024            0.051716                      0.051716                   yes
 38    0.2  1      3.254     